## STEP: 9 T-Test — Delivery & Customer Satisfaction


## Connect the SQL Database

In [1]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="MySQL@2026Strong",
    database="cart2"
)

cursor = conn.cursor()

print("Connected successfully!")
print(conn.is_connected())

Connected successfully!
True


In [2]:
query = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.actual_delivery_days,
    o.delivery_delay_days,
    oi.order_value,
    r.review_score
FROM orders o

LEFT JOIN (
    SELECT
        order_id,
        ROUND(SUM(price + freight_value), 2) AS order_value
    FROM order_items
    GROUP BY order_id
) oi
    ON o.order_id = oi.order_id

LEFT JOIN (
    SELECT
        order_id,
        ROUND(AVG(review_score), 2) AS review_score
    FROM order_reviews
    GROUP BY order_id
) r
    ON o.order_id = r.order_id;
"""

eda_df = pd.read_sql(query, conn)

print("Shape:", eda_df.shape)

C:\Users\VIDYAPRIYA V\AppData\Local\Temp\ipykernel_7600\2786236258.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  eda_df = pd.read_sql(query, conn)


Shape: (99441, 8)


## Create the two T-test groups

In [3]:
test_df = eda_df[
    ['delivery_delay_days', 'review_score']
].dropna()

delayed_reviews = test_df.loc[
    test_df['delivery_delay_days'] > 0,
    'review_score'
]

on_time_reviews = test_df.loc[
    test_df['delivery_delay_days'] <= 0,
    'review_score'
]

print("Delayed orders:", len(delayed_reviews))
print("On-time / early orders:", len(on_time_reviews))

print("\nDelayed mean review score:",
      round(delayed_reviews.mean(), 3))

print("On-time / early mean review score:",
      round(on_time_reviews.mean(), 3))

Delayed orders: 6382
On-time / early orders: 89448

Delayed mean review score: 2.272
On-time / early mean review score: 4.29


The group means are clearly indicates:

Delayed:       6,382 orders  → mean review 2.272
On-time/early: 89,448 orders → mean review 4.290

## Perform the T-Test

In [4]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(
    delayed_reviews,
    on_time_reviews,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: -100.74484590688685
P-value: 0.0


In [ ]:
The T-test result is unambiguous. One minor technical clarification: the p-value reported as 0.0 by SciPy does not represent 
a literal mathematical zero — it indicates that the true p-value is smaller than can be displayed within standard floating-point precision.
Accordingly, this should be reported as p < 0.001 rather than p = 0.0.


## Step 5 — Check the T-test assumptions

In [ ]:
For an independent two-sample t-test, several assumptions must be considered:

Independence: Each order is treated as an independent observation.
Continuous numerical outcome: Review score is numerical in nature, though it is bounded and discrete (ranging from 1 to 5), 
meaning this assumption is not fully satisfied.
Normality: Given the large sample sizes involved (6,382 and 89,448 observations), the t-test remains generally robust to moderate deviations 
from normality.
Equal variance: Welch's t-test (equal_var=False) was applied, which does not require the assumption of equal variances between groups.

## Step 6 — Statistical Summary

In [ ]:
| Measure                     |   Result |
| --------------------------- | -------: |
| Delayed orders              |    6,382 |
| On-time / early orders      |   89,448 |
| Delayed mean rating         |    2.272 |
| On-time / early mean rating |    4.290 |
| T-statistic                 | −100.745 |
| P-value                     |  < 0.001 |
| Significance level (α)      |     0.05 |
Since:

p < 0.05

we reject H₀.


## Step 7 — Business Insight

In [ ]:
There is a statistically significant difference in review scores between delayed and on-time/early orders. 
In this dataset, delayed orders have a much lower average review score (2.27) than on-time/early orders (4.29).

This provides strong statistical evidence of an association between delivery status and customer satisfaction. 
It does not by itself prove that delays cause lower ratings.

In [ ]:
## T-Test — Delivery & Customer Satisfaction

### Business Question

Do delayed orders receive significantly lower review scores compared with orders delivered on time or early?

### Hypotheses

* **H₀ (Null Hypothesis):** There is no significant difference in the mean review scores between delayed and on-time/early orders.
* **H₁ (Alternative Hypothesis):** There is a significant difference in the mean review scores between delayed and on-time/early orders.

### Method

An **independent two-sample Welch's t-test** was used to compare the mean review scores of the two groups. Welch's test was selected because it does not require the two groups to have equal variances.

### Results

| Metric            | Delayed | On Time / Early |
| ----------------- | ------: | --------------: |
| Number of orders  |   6,382 |          89,448 |
| Mean review score |   2.272 |           4.290 |

The test produced:

* **T-statistic:** −100.745
* **P-value:** < 0.001
* **Significance level:** 0.05

### Statistical Decision

Because the p-value is less than 0.05, the null hypothesis is **rejected**.

### Business Insight

The analysis shows a statistically significant difference in customer review scores between delayed and on-time/early orders. 
Delayed orders have a substantially lower average review score (2.27) compared with on-time/early orders (4.29).

This indicates a strong statistical association between delivery status and customer satisfaction in the analyzed dataset. 
However, the t-test alone does not establish that delivery delays directly cause lower review scores.


In [ ]:
The One-Way ANOVA — Product Category & Spending using your project definition of order value:
Order value = price + freight_value
Because an order can contain products from multiple categories, we’ll calculate category-level spending within each order. 
This avoids assigning the entire order value to every category.

## Step 1 — Create the ANOVA dataset

In [7]:
anova_df = pd.read_sql("""
SELECT
    COALESCE(p.product_category_name_english, 'Unknown') AS category,
    oi.order_id,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS category_order_value
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY
    COALESCE(p.product_category_name_english, 'Unknown'),
    oi.order_id
""", conn)

print("Shape:", anova_df.shape)
print(anova_df.head())

C:\Users\VIDYAPRIYA V\AppData\Local\Temp\ipykernel_14920\3487561845.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  anova_df = pd.read_sql("""


Shape: (99470, 3)
         category                          order_id  category_order_value
0       perfumery  f30149f4a8882a08895b6a242aa0d612                120.24
1            auto  f5eda0ded77c1293b04c953138c8331d                143.83
2  bed_bath_table  0bf736fd0fd5169d60de3699fcbcf986                242.10
3      housewares  3aba44d8e554ab4bb8c09f6f78032ca8                 78.50
4      housewares  6f0dfb5b5398b271cc6bbd9ee263530e                 78.50


In [8]:
anova_summary = (
    anova_df
    .groupby('category')
    .agg(
        observations=('category_order_value', 'count'),
        mean_spending=('category_order_value', 'mean')
    )
    .sort_values('mean_spending', ascending=False)
)

print(anova_summary)

                                       observations  mean_spending
category                                                          
computers                                       181    1286.184696
small_appliances_home_oven_and_coffee            75     669.247600
home_appliances_2                               234     529.563761
agro_industry_and_commerce                      182     430.626758
musical_instruments                             628     334.613646
...                                             ...            ...
electronics                                    2550      81.107867
cds_dvds_musicals                                12      79.582500
diapers_and_hygiene                              27      79.306296
flowers                                          29      55.134828
home_comfort_2                                   24      48.774167

[72 rows x 2 columns]


## Step 2 — Check ANOVA assumptions

In [ ]:
On one-way ANOVA, we mainly check:

Independence —            observations should be reasonably independent.
Normality —               each group's spending should be approximately normally distributed.
Homogeneity of variance — group variances should be reasonably similar.

Because spending data is typically right-skewed, with many categories and unequal sample sizes,
we'll use Levene's test for variance equality and inspect the distributions rather than relying only on a normality test.



In [9]:
from scipy.stats import levene
import numpy as np

# Create groups
groups = [
    group['category_order_value'].dropna().values
    for _, group in anova_df.groupby('category')
]

# Levene's test for equality of variances
levene_stat, levene_p = levene(*groups, center='median')

print("Levene statistic:", levene_stat)
print("Levene p-value:", levene_p)

if levene_p < 0.05:
    print("Conclusion: Variances are significantly different.")
else:
    print("Conclusion: No significant evidence of unequal variances.")

Levene statistic: 60.873992667072166
Levene p-value: 0.0
Conclusion: Variances are significantly different.


In [ ]:
The result shows that the equal-variance assumption is violated:

Levene statistic: 60.874
p-value: < 0.001
Therefore, the category variances are significantly different.

To this project,perform the One-Way ANOVA, but  clearly mention this assumption violation. Since the groups are also highly unequal in size, the result should be interpreted with caution.

## Step 3 — Perform One-Way ANOVA

In [10]:
from scipy.stats import f_oneway

f_stat, p_value = f_oneway(*groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

alpha = 0.05

if p_value < alpha:
    print("Decision: Reject H0")
    print("Conclusion: Average spending differs significantly across product categories.")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: No significant difference in average spending across product categories.")

F-statistic: 153.7961218357648
P-value: 0.0
Decision: Reject H0
Conclusion: Average spending differs significantly across product categories.


In [ ]:
| Statistic              |        Result |
| ---------------------- | ------------: |
| Number of categories   |            72 |
| F-statistic            |   **153.796** |
| p-value                |   **< 0.001** |
| Significance level (α) |          0.05 |
| Decision               | **Reject H₀** |


In [ ]:
The displayed value of 0.0 should be reported as p < 0.001, since the actual p-value is smaller than the numerical precision being displayed 
rather than being literally zero.

Statistical Conclusion

Since p < 0.05, the null hypothesis is rejected.

There is statistically significant evidence that average spending differs across product categories.

It is important to note that ANOVA indicates at least one category mean differs from the others — it does not identify which specific categories 
differ from one another. A post-hoc test, such as Tukey's HSD, would be required to determine that.

Assumption Limitation

Levene's test produced the following results:

Statistic = 60.874
p < 0.001

This indicates that the equal-variance assumption was violated. Additionally, the category sample sizes are highly unequal.

As a result, the standard ANOVA finding should be presented alongside this limitation. While the very small p-value provides strong evidence of 
differences in means, the heterogeneity of variances means the result should not be interpreted as though all classical ANOVA assumptions were 
fully satisfied.

Business Interpretation

The analysis indicates that customer spending varies considerably by product category. The descriptive results illustrate this clearly: 
the computers category averaged approximately ₹1,286 in order spending, compared to approximately ₹49 for home_comfort_2.

This suggests that product category is associated with substantially different spending levels — an insight that can inform pricing strategy, 
category planning, product assortment decisions, and broader revenue strategy.

In [ ]:
## ANOVA: Product Category and Spending

### Objective

To determine whether average spending differs significantly across different product categories.

### Hypotheses

**Null Hypothesis (H₀):**
The mean order-category spending is the same across all product categories.

**Alternative Hypothesis (H₁):**
At least one product category has a significantly different mean order-category spending.

### Method

A **One-Way ANOVA** was performed using:

* **Independent variable:** Product category
* **Dependent variable:** Order-category spending
* **Number of categories:** 72
* **Significance level:** α = 0.05

Order-category spending was calculated as the sum of `price + freight_value` for each order within each product category.

### Assumption Check

Levene's test was used to check the equality of variances.

* Levene statistic = **60.874**
* p-value < **0.001**

Since p < 0.05, the equal-variance assumption was violated. The category sample sizes were also unequal. Therefore, the ANOVA result should be 
interpreted with caution.

### ANOVA Results

| Measure     |        Result |
| ----------- | ------------: |
| F-statistic |   **153.796** |
| p-value     |   **< 0.001** |
| α           |      **0.05** |
| Decision    | **Reject H₀** |

### Statistical Decision

Since the p-value is less than 0.05, the null hypothesis is rejected.

There is statistically significant evidence that average spending differs across product categories.

The ANOVA result indicates that at least one product category has a different mean spending level. ANOVA alone does not identify which specific 
categories differ.

### Business Insight

The results indicate substantial variation in customer spending across product categories. For example, the descriptive analysis showed that the `
computers` category had considerably higher average category-order spending than lower-spending categories such as `home_comfort_2`.

This information can support category-level pricing analysis, product assortment decisions, revenue planning, and targeted business strategies.

### Limitation

The equal-variance assumption was violated, as indicated by Levene's test. In addition, an order containing products from multiple categories 
can contribute observations to more than one category. Therefore, the observations are not perfectly independent across categories. The ANOVA result 
should consequently be interpreted as evidence of differences in category-level spending rather than as definitive causal evidence that product 
category causes higher or lower spending.


## Chi-Square Test — Payment Method & Order Status

In [11]:
chi_df = pd.read_sql("""
SELECT
    op.payment_type,
    o.order_status,
    COUNT(*) AS order_count
FROM order_payments op
JOIN orders o
    ON op.order_id = o.order_id
GROUP BY
    op.payment_type,
    o.order_status
ORDER BY
    op.payment_type,
    o.order_status
""", conn)

print(chi_df)

C:\Users\VIDYAPRIYA V\AppData\Local\Temp\ipykernel_14920\1920715110.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  chi_df = pd.read_sql("""


   payment_type order_status  order_count
0        boleto     canceled           95
1        boleto      created            2
2        boleto    delivered        19191
3        boleto     invoiced           67
4        boleto   processing           70
5        boleto      shipped          209
6        boleto  unavailable          150
7   credit_card     approved            2
8   credit_card     canceled          444
9   credit_card      created            3
10  credit_card    delivered        74586
11  credit_card     invoiced          239
12  credit_card   processing          224
13  credit_card      shipped          851
14  credit_card  unavailable          446
15   debit_card     canceled            7
16   debit_card    delivered         1486
17   debit_card     invoiced            6
18   debit_card   processing            2
19   debit_card      shipped           22
20   debit_card  unavailable            6
21  not_defined     canceled            3
22      voucher     canceled      

## Step 2 — Check multiple payment methods per order

In [12]:
multi_payment_orders = pd.read_sql("""
SELECT
    order_id,
    COUNT(DISTINCT payment_type) AS payment_method_count
FROM order_payments
GROUP BY order_id
HAVING COUNT(DISTINCT payment_type) > 1
""", conn)

print("Orders with multiple payment methods:",
      len(multi_payment_orders))

Orders with multiple payment methods: 2246


C:\Users\VIDYAPRIYA V\AppData\Local\Temp\ipykernel_14920\3784790803.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  multi_payment_orders = pd.read_sql("""


In [ ]:
2,246 orders use more than one payment method, so we should avoid pretending that every order has a single payment method.

For the project, we can use the payment method records as the unit of analysis, which matches original SQL output.


## Step 3 — Create the contingency table

In [13]:
contingency_table = chi_df.pivot_table(
    index='payment_type',
    columns='order_status',
    values='order_count',
    fill_value=0
)

print(contingency_table)

order_status  approved  canceled  created  delivered  invoiced  processing  \
payment_type                                                                 
boleto             0.0      95.0      2.0    19191.0      67.0        70.0   
credit_card        2.0     444.0      3.0    74586.0     239.0       224.0   
debit_card         0.0       7.0      0.0     1486.0       6.0         2.0   
not_defined        0.0       3.0      0.0        0.0       0.0         0.0   
voucher            0.0     115.0      0.0     5493.0      13.0        23.0   

order_status  shipped  unavailable  
payment_type                        
boleto          209.0        150.0  
credit_card     851.0        446.0  
debit_card       22.0          6.0  
not_defined       0.0          0.0  
voucher          84.0         47.0  


## Step 4 — Check the expected frequencies

In [14]:
from scipy.stats import chi2_contingency

chi2_stat, p_value, dof, expected = chi2_contingency(
    contingency_table
)

expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns
)

print("Chi-square statistic:", chi2_stat)
print("Degrees of freedom:", dof)
print("P-value:", p_value)

print("\nExpected frequencies:")
print(expected_df)

print("\nMinimum expected frequency:",
      expected.min())

print("\nExpected cells below 5:",
      (expected < 5).sum())

print("Total cells:",
      expected.size)

Chi-square statistic: 677.0831369637666
Degrees of freedom: 28
P-value: 1.2047397488650834e-124

Expected frequencies:
order_status  approved    canceled   created     delivered    invoiced  \
payment_type                                                             
boleto        0.380879  126.451841  0.952198  19187.924302   61.892844   
credit_card   1.478448  490.844580  3.696119  74481.229617  240.247723   
debit_card    0.029436    9.772789  0.073590   1482.932484    4.783368   
not_defined   0.000058    0.019175  0.000144      2.909612    0.009385   
voucher       0.111180   36.911615  0.277949   5601.003985   18.066679   

order_status  processing     shipped  unavailable  
payment_type                                       
boleto         60.750207  222.052481   123.595249  
credit_card   235.812381  861.934909   479.756223  
debit_card      4.695060   17.161254     9.552019  
not_defined     0.009212    0.033672     0.018742  
voucher        17.733140   64.817685    36.077768 

## Step 5 — Interpret the Chi-Square assumption check

In [ ]:
Results:

Total cells: 40
Expected cells < 5: 18
Percentage: 45%
Minimum expected frequency: 0.000058

So the standard Chi-Square assumptions are not satisfied. More than 20% of the expected cells are below 5, largely because of the very sparse approved,
created, and especially not_defined categories.

However, the calculated test gives:

χ² = 677.083
df = 28
p = 1.20 × 10⁻¹²⁴, therefore p < 0.001

The extremely small p-value indicates a very strong departure from independence for this payment-record contingency table, but because the 
expected-frequency assumption is violated, we should not rely on this result alone.

## Step 6 — Handle the sparse categories

In [ ]:
For a project report, a practical approach would be to combine the very rare order statuses:

approved
created
processing
invoiced
shipped
unavailable
canceled
delivered

That said, categories should not be combined arbitrarily simply to achieve statistical significance. Instead, grouping should be based on a meaningful
business distinction, such as:

Completed: delivered
Not completed: canceled, unavailable, processing, invoiced, shipped, created, approved

However, it is worth noting that statuses such as shipped and processing represent different stages of fulfillment, so this grouping does shift the
nature of the original question somewhat.

For this assignment, the recommended approach is to report the original Chi-Square result alongside its assumption limitation, rather than obscuring
the underlying issue.

Final Statistical Conclusion

At α = 0.05:

H₀: Payment method and order status are independent.
H₁: Payment method and order status are associated.

The calculated p-value is < 0.001, providing strong statistical evidence against independence. However, since 45% of expected cell counts fall below 
5, the standard Chi-Square approximation is unreliable, and the result should therefore be interpreted with caution.

In [ ]:
## Chi-Square Test: Payment Method and Order Status

### Objective

To determine whether payment method is significantly associated with order status.

### Hypotheses

**Null Hypothesis (H₀):**
Payment method and order status are independent.

**Alternative Hypothesis (H₁):**
Payment method and order status are significantly associated.

### Method

A **Chi-Square Test of Independence** was performed using:

* **Independent variable:** Payment method
* **Outcome variable:** Order status
* **Significance level:** α = 0.05

The contingency table was constructed using payment records joined with order status.

### Assumption Check

The Chi-Square test requires sufficiently large expected frequencies.

The analysis produced:

* Total expected cells = **40**
* Expected cells below 5 = **18**
* Percentage below 5 = **45%**
* Minimum expected frequency = **0.000058**

Therefore, the expected-frequency assumption was violated. The sparse cells are mainly associated with rare order statuses and the 
`not_defined` payment category.

### Test Results

| Measure              |      Result |
| -------------------- | ----------: |
| Chi-Square statistic | **677.083** |
| Degrees of freedom   |      **28** |
| p-value              | **< 0.001** |
| Significance level   |    **0.05** |

### Statistical Decision

The p-value is less than 0.05, so the null hypothesis is rejected.

The analysis provides strong statistical evidence of an association between payment method and order status.

However, because 45% of expected cells are below 5, the standard Chi-Square approximation is not fully reliable. Therefore, this finding should be 
interpreted with caution rather than treated as definitive.

### Business Insight

The results suggest that payment methods and order outcomes are not distributed identically in the dataset. This may help identify differences 
in order-status patterns across payment methods and can support further investigation of payment and fulfillment processes.

### Important Data Limitation

There are **2,246 orders with multiple payment methods**. Therefore, the analysis uses payment records rather than strictly one payment method per 
order. Consequently, the result should be interpreted as an association between **payment-record type and order status**, rather than a perfect 
one-to-one relationship between an order's single payment method and its final status.


## Step 10 — Statistical Analysis Summary

In [ ]:
Summary of Statistical Tests

Test	                      Question	                                   Statistic	       p-value	    Decision	           Conclusion
T-Test	               Do delayed orders receive lower reviews?	           −100.745	            <0.001	     Reject H₀	   Significant difference
ANOVA	               Does spending differ by category?	                153.796	            <0.001	     Reject H₀	   Significant difference
Chi-Square	           Are payment method and status associated?	        677.083	            <0.001*	     Reject H₀     *Evidence of association*

*The Chi-Square test shows a substantial violation of the expected-frequency assumption, so this result should be interpreted with caution.

## Overall Interpretation

The statistical analysis reveals three notable patterns within the dataset:

Delivery performance and customer satisfaction: Delayed orders are associated with substantially lower review scores.
Product category and spending: Average spending differs significantly across product categories.
Payment method and order status: The distribution of payment records varies across order statuses, though the assumption violation in the 
Chi-Square test limits the strength of this conclusion.
These findings represent statistical associations and should not be interpreted as evidence of causation.